In [7]:
#1
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
!pip uninstall -y peft
!pip install peft==0.10.0

Found existing installation: peft 0.10.0
Uninstalling peft-0.10.0:
  Successfully uninstalled peft-0.10.0
  Using cached peft-0.10.0-py3-none-any.whl.metadata (13 kB)
Using cached peft-0.10.0-py3-none-any.whl (199 kB)


In [9]:
!pip install transformers datasets accelerate evaluate sacrebleu evaluate

In [10]:
#5
import os
os.listdir()

['.config', 'dogri-datasetv02.csv', 'drive', 'sample_data']

In [11]:
#7
import pandas as pd
df = pd.read_csv("dogri-datasetv02.csv")
df.head()

,src_dogri,tgt_english
0,मेरे कोल आ,come to me
1,जा,go
2,इद्दर बेई जा,sit here
3,खलोई आ,stand up
4,बलगी जा,wait


In [12]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, TrainingArguments, Trainer
from datasets import load_dataset
import evaluate

In [13]:
#8
from datasets import load_dataset

dataset = load_dataset("csv", data_files="dogri-datasetv02.csv")
dataset = dataset["train"].train_test_split(test_size=0.2, seed=42)

Generating train split: 0 examples [00:00, ? examples/s]

In [14]:
#10
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

tokenizer.src_lang = "doi_Deva"
tokenizer.tgt_lang = "eng_Latn"

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [15]:
print(model)

M2M100ForConditionalGeneration(
  (model): M2M100Model(
    (shared): M2M100ScaledWordEmbedding(256206, 1024, padding_idx=1)
    (encoder): M2M100Encoder(
      (embed_tokens): M2M100ScaledWordEmbedding(256206, 1024, padding_idx=1)
      (embed_positions): M2M100SinusoidalPositionalEmbedding()
      (layers): ModuleList(
        (0-11): 12 x M2M100EncoderLayer(
          (self_attn): M2M100Attention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): ReLU()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
       

In [16]:
#11
def preprocess(example):
    inputs = tokenizer(
        example["src_dogri"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

    targets = tokenizer(
        example["tgt_english"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

    inputs["labels"] = targets["input_ids"]
    return inputs

dataset = dataset.map(preprocess)

Map:   0%|          | 0/120 [00:00<?, ? examples/s]

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

In [17]:
#12
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_2_SEQ_LM"
)

model = get_peft_model(model, lora_config)

In [18]:
#13
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=30,
    save_strategy="no",
    logging_steps=10
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"]
)

trainer.train()

Step,Training Loss
10,14.098323
20,14.057397
30,13.809874
40,13.735211
50,13.459987
60,13.300497
70,13.114415
80,12.978499
90,12.881183
100,12.733708


TrainOutput(global_step=900, training_loss=10.180955632527668, metrics={'train_runtime': 381.3185, 'train_samples_per_second': 9.441, 'train_steps_per_second': 2.36, 'total_flos': 1703817510912000.0, 'train_loss': 10.180955632527668, 'epoch': 30.0})

In [19]:
#14
def translate(text):
    # This is the "no-fail" way to get the language ID
    target_lang = "eng_Latn"
    forced_bos_token_id = tokenizer.convert_tokens_to_ids(target_lang)

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    output = model.generate(
        **inputs,
        forced_bos_token_id=forced_bos_token_id,
        max_new_tokens=64
    )

    return tokenizer.decode(output[0], skip_special_tokens=True)

In [20]:
#15
predictions = []
references = []
dogri_sentences = []

for example in dataset["test"]:
    pred = translate(example["src_dogri"])
    predictions.append(pred)
    references.append(example["tgt_english"])
    dogri_sentences.append(example["src_dogri"])

In [21]:
import evaluate

# BLEU (baseline)
bleu = evaluate.load("bleu")
bleu_score = bleu.compute(
    predictions=predictions,
    references=[[r] for r in references]
)

print("BLEU:", bleu_score["bleu"])


# METEOR (MAIN METRIC)
meteor = evaluate.load("meteor")
meteor_score = meteor.compute(
    predictions=predictions,
    references=references
)

print("METEOR:", meteor_score["meteor"])

BLEU: 0.07627708496558526


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


METEOR: 0.44600252667964946


In [22]:
#18
# Check the first pair
print(f"Pred: {predictions[7]}")
print(f"Ref: {references[7]}")

# Check types - References should usually be a list of lists
print(f"Type Pred: {type(predictions[7])}")
print(f"Type Ref: {type(references[7])}")

Pred: I'm going home
Ref: girl is going home
Type Pred: <class 'str'>
Type Ref: <class 'str'>


In [23]:
print("BLEU: ", bleu_score["bleu"])
print("METEOR:", meteor_score["meteor"])

BLEU:  0.07627708496558526
METEOR: 0.44600252667964946


In [24]:
import json

scores = {
    "bleu": bleu_score,
    "meteor": meteor_score
}

with open("/content/drive/MyDrive/scores.json", "w") as f:
    json.dump(scores, f)

In [25]:
model.save_pretrained("/content/drive/MyDrive/dogri2_model")
tokenizer.save_pretrained("/content/drive/MyDrive/dogri2_model")

('/content/drive/MyDrive/dogri2_model/tokenizer_config.json',
 '/content/drive/MyDrive/dogri2_model/tokenizer.json')